# CF-HPINO Demo Notebook

Walkthrough: YAML config, synthetic data, hybrid loss, short training, baseline comparison.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt

from src.utils.config_loader import load_experiment, build_trainer_from_experiment
from src.data import OptionPricingDataset
from src.data.sampling import collate_option_batch
from src.eval.metrics import compare_methods, results_to_table
from src.eval.plots import plot_surface_from_grid, plot_convergence, plot_error_heatmap, plot_comparison_bar
from src.baselines.pinn import build_pinn
from src.baselines.fno_baseline import build_pure_fno

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| Device:", device)

In [ ]:
exp = load_experiment(ROOT / "configs" / "smoke_test.yaml")
exp["train"].device = str(device)
model, loss_fn, trainer = build_trainer_from_experiment(exp, device=str(device))
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import DataLoader

ds = OptionPricingDataset(exp["data"])
loader = DataLoader(ds, batch_size=2, collate_fn=collate_option_batch)
batch = next(iter(loader))
plot_surface_from_grid(ds[0]["grid"], ds[0]["price_surface"], title="BS surface (sample 0)")
plt.show()

In [ ]:
model, loss_fn = model.to(device), loss_fn.to(device)
batch = {k: v.to(device) for k, v in batch.items()}
loss, breakdown = loss_fn(model, batch, return_breakdown=True)
print("Initial loss:", f"{loss.item():.4e}")
print(breakdown)

In [ ]:
history = trainer.train()
plot_convergence(history)
plt.show()

In [ ]:
results = compare_methods(
    {"CF-HPINO": model, "PureFNO": build_pure_fno().to(device), "PINN": build_pinn().to(device)},
    batch["params"], batch["coords"], batch["prices"], device=str(device),
)
print(results_to_table(results))
plot_comparison_bar({k: v.relative_l2 for k, v in results.items()})
plt.show()

In [ ]:
with torch.no_grad():
    pred = model(batch["params"][:1], batch["coords"][:1])
plot_error_heatmap(batch["grid"][:1], pred.view(-1), batch["prices"][:1].view(-1))
plt.show()